# 11 · Approximate Nearest Neighbor (ANN) Scaling Evaluation
## Exact search (FlatL2) vs IVF vs HNSW, at a realistic corpus scale

**Why this notebook exists:** the thesis proposal frames the core method as
*"embedding-based semantic retrieval combined with approximate nearest neighbor
search"* — but every retrieval notebook so far (01-07, 09b, 10) uses **exact**
brute-force search (`faiss.IndexFlatL2`/`IndexFlatIP`), never true ANN. That's
been fine at 98,716 companies (exact search is already sub-millisecond-to-a-few-ms
there), but it means the thesis's own stated core technique has never actually
been tested. This notebook fills that gap.

**The corpus-scale problem, and how this notebook handles it honestly:**
We only have 98,716 real, labeled companies available locally (no larger dataset
exists on disk, and pulling millions more from the API isn't feasible given the
current quota). Exact search doesn't get meaningfully slower until an index is
much bigger than that. So this notebook builds a **synthetic-scale index**:

```
Real signal:      98,716 real MiniLM company embeddings (from 03_baseline_minilm)
Synthetic filler: ~1.9M additional vectors, each generated by taking a random
                   REAL embedding and adding small Gaussian noise -- these sit in
                   the same semantic neighborhoods as real companies (harder,
                   more realistic distractors than pure random noise would be),
                   but are NOT real companies and can never match ground truth
Combined index:   ~2,000,000 vectors total (configurable via TARGET_CORPUS_SIZE)
```

**What this can and cannot tell us:**
- ✅ Query **latency** at a realistic ~2M-vector scale, for exact vs IVF vs HNSW —
  this is real and meaningful, since latency depends on index size and structure,
  not on whether the vectors are "real" companies.
- ✅ Retrieval **quality** (NDCG/Precision/Recall/F1) relative to ground truth,
  since synthetic vectors can never be relevant (ground truth only contains real,
  labeled domains) — this tests whether ANN's approximation still finds the real
  companies once the index is padded with realistic-looking distractors.
- ❌ This is NOT a test of quality on an independently-sourced, truly 20M-scale
  corpus — the synthetic filler is derived from the same 98,716 real embeddings'
  distribution, not from 20M more real, distinct companies. Treat quality numbers
  here as "does ANN preserve exact search's quality once the index is padded to
  realistic scale," not "this is what would happen on the real full GOI."

**Folder structure:**
```
result/
└── 11_ann_scaling/
    ├── synthetic_embeddings.npy      # Cached ~1.9M synthetic filler vectors
    ├── combined_embeddings.npy       # Real + synthetic, concatenated (cached)
    ├── index_flat.faiss              # Exact baseline index at full scale
    ├── index_ivf.faiss               # IVF index
    ├── index_hnsw.faiss              # HNSW index
    ├── evaluation_ann.csv            # NDCG/Prec/Recall/F1 per index config
    ├── latency_memory_summary.csv    # Latency + memory footprint per config
    └── (printed key findings)
```

## 1 · Environment Setup

In [7]:
import time, json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
import torch
from dotenv import load_dotenv

load_dotenv(override=True)

RESULT_DIR = Path('result/11_ann_scaling')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MINILM_EMB_PATH = Path('result/03_baseline_minilm/company_embeddings.npy')

print(f'[Setup] Result folder : {RESULT_DIR}/ — ready')
print(f'[Setup] Real embeddings source : {MINILM_EMB_PATH}')

[Setup] Result folder : result/11_ann_scaling/ — ready
[Setup] Real embeddings source : result/03_baseline_minilm/company_embeddings.npy


## 2 · Load Real Company Embeddings

Reuses the MiniLM embeddings already computed and cached by
`03_baseline_minilm.ipynb` — no need to re-encode the corpus.

In [8]:
print('[Load] Loading real MiniLM company embeddings...')
real_embeddings = np.load(MINILM_EMB_PATH).astype('float32')
N_REAL = real_embeddings.shape[0]
DIM    = real_embeddings.shape[1]
print(f'[Load] Real embeddings shape : {real_embeddings.shape}')

print('[Load] Loading corpus metadata (for domain lookup on real indices only)...')
all_companies = pd.read_excel('dataset/production_results.xlsx')
all_companies = all_companies.drop_duplicates(subset='domain').reset_index(drop=True)
assert len(all_companies) == N_REAL, (
    f'Corpus metadata ({len(all_companies)} rows) does not match embeddings '
    f'({N_REAL} rows) -- these must be in the same order as when '
    f'03_baseline_minilm.ipynb encoded them.'
)
print(f'[Load] Corpus metadata rows  : {len(all_companies):,} (matches embeddings ✅)')

[Load] Loading real MiniLM company embeddings...
[Load] Real embeddings shape : (98716, 384)
[Load] Loading corpus metadata (for domain lookup on real indices only)...
[Load] Corpus metadata rows  : 98,716 (matches embeddings ✅)


## 3 · Generate Synthetic Filler Vectors

Each synthetic vector = a random REAL embedding + small Gaussian noise (scaled
to a fraction of the corpus's typical embedding norm). This places synthetic
vectors in the same semantic neighborhoods as real companies -- a harder, more
realistic distractor than pure random noise, which would be trivially easy for
any index (exact or approximate) to distinguish from real signal.

Cached to disk after first generation, since this is a fixed, reusable
"scale-up" of the corpus that doesn't need to be regenerated every run.

In [9]:
TARGET_CORPUS_SIZE = 2_000_000  # total vectors (real + synthetic) -- adjust down if
                                  # index build time is too long for your SLURM time limit,
                                  # or up if you want a closer approximation to the real ~20M GOI
NOISE_SCALE = 0.15  # fraction of the average real embedding norm used as Gaussian noise std

N_SYNTHETIC = max(0, TARGET_CORPUS_SIZE - N_REAL)
synthetic_path = RESULT_DIR / 'synthetic_embeddings.npy'

print(f'[Synthetic] Target corpus size : {TARGET_CORPUS_SIZE:,}')
print(f'[Synthetic] Real vectors       : {N_REAL:,}')
print(f'[Synthetic] Synthetic needed   : {N_SYNTHETIC:,}')

if synthetic_path.exists():
    print('[Synthetic] Loading cached synthetic vectors...')
    synthetic_embeddings = np.load(synthetic_path).astype('float32')
    if synthetic_embeddings.shape[0] != N_SYNTHETIC:
        print(f'[Synthetic] WARNING: cached count ({synthetic_embeddings.shape[0]:,}) does not '
              f'match N_SYNTHETIC ({N_SYNTHETIC:,}) -- regenerating.')
        synthetic_path.unlink()

if not synthetic_path.exists():
    print('[Synthetic] Generating synthetic filler vectors...')
    rng = np.random.default_rng(seed=42)
    avg_norm = np.linalg.norm(real_embeddings, axis=1).mean()
    noise_std = avg_norm * NOISE_SCALE
    print(f'[Synthetic] Average real embedding norm : {avg_norm:.4f}')
    print(f'[Synthetic] Gaussian noise std           : {noise_std:.4f}')

    t0 = time.time()
    source_idx = rng.integers(0, N_REAL, size=N_SYNTHETIC)
    noise = rng.normal(loc=0.0, scale=noise_std, size=(N_SYNTHETIC, DIM)).astype('float32')
    synthetic_embeddings = real_embeddings[source_idx] + noise
    print(f'[Synthetic] Generated {N_SYNTHETIC:,} vectors in {time.time()-t0:.1f}s')

    np.save(synthetic_path, synthetic_embeddings)
    print(f'[Synthetic] Saved to {synthetic_path}')

print(f'[Synthetic] Final synthetic shape : {synthetic_embeddings.shape}')

[Synthetic] Target corpus size : 2,000,000
[Synthetic] Real vectors       : 98,716
[Synthetic] Synthetic needed   : 1,901,284
[Synthetic] Loading cached synthetic vectors...
[Synthetic] Final synthetic shape : (1901284, 384)


In [10]:
combined_path = RESULT_DIR / 'combined_embeddings.npy'

if combined_path.exists():
    print('[Combined] Loading cached combined (real + synthetic) embeddings...')
    combined_embeddings = np.load(combined_path).astype('float32')
else:
    print('[Combined] Concatenating real + synthetic embeddings...')
    combined_embeddings = np.concatenate([real_embeddings, synthetic_embeddings], axis=0).astype('float32')
    np.save(combined_path, combined_embeddings)
    print(f'[Combined] Saved to {combined_path}')

N_TOTAL = combined_embeddings.shape[0]
print(f'[Combined] Combined shape : {combined_embeddings.shape}')
print(f'[Combined] Indices [0, {N_REAL}) are REAL companies; '
      f'[{N_REAL}, {N_TOTAL}) are synthetic filler (never relevant).')

[Combined] Loading cached combined (real + synthetic) embeddings...
[Combined] Combined shape : (2000000, 384)
[Combined] Indices [0, 98716) are REAL companies; [98716, 2000000) are synthetic filler (never relevant).


## 4 · Build Indexes: Exact (FlatL2), IVF, HNSW

All three indexes are built over the SAME combined (real + synthetic) vector
set, so any latency/quality difference between them is attributable to the
indexing method, not to different data.

- **Exact (`IndexFlatL2`)** — the baseline used everywhere else in this thesis;
  brute-force, always finds the true nearest neighbours, but scales linearly
  with corpus size.
- **IVF (`IndexIVFFlat`)** — clusters vectors into `nlist` cells at build time;
  at query time, only searches the `nprobe` nearest cells instead of everything.
  `nprobe` is the single tunable knob explored below.
- **HNSW (`IndexHNSWFlat`)** — a navigable small-world graph; `efSearch` controls
  how much of the graph is explored per query. FAISS's HNSW implementation is
  CPU-only (no GPU support), so this index is built and searched on CPU
  regardless of GPU availability.

In [11]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[Index] GPU available : {torch.cuda.is_available()}')

# ── Exact FlatL2 (baseline) ────────────────────────────────────────────────
flat_path = RESULT_DIR / 'index_flat.faiss'
if flat_path.exists():
    print('[Index:Flat] Loading cached exact index...')
    index_flat = faiss.read_index(str(flat_path))
    FLAT_BUILD_TIME = None
else:
    print('[Index:Flat] Building exact IndexFlatL2...')
    t0 = time.time()
    index_flat = faiss.IndexFlatL2(DIM)
    index_flat.add(combined_embeddings)
    FLAT_BUILD_TIME = time.time() - t0
    faiss.write_index(index_flat, str(flat_path))
    print(f'[Index:Flat] Built in {FLAT_BUILD_TIME:.1f}s, {index_flat.ntotal:,} vectors')

[Index] GPU available : False
[Index:Flat] Loading cached exact index...


In [12]:
# ── IVF ──────────────────────────────────────────────────────────────────────
IVF_NLIST = 4096  # standard heuristic ballpark for ~1-2M vectors (roughly 4*sqrt(N))

ivf_path = RESULT_DIR / 'index_ivf.faiss'
if ivf_path.exists():
    print('[Index:IVF] Loading cached IVF index...')
    index_ivf = faiss.read_index(str(ivf_path))
    IVF_BUILD_TIME = None
else:
    print(f'[Index:IVF] Building IndexIVFFlat (nlist={IVF_NLIST})...')
    t0 = time.time()
    quantizer = faiss.IndexFlatL2(DIM)
    index_ivf = faiss.IndexIVFFlat(quantizer, DIM, IVF_NLIST)
    print('[Index:IVF] Training (k-means clustering)...')
    index_ivf.train(combined_embeddings)
    print('[Index:IVF] Adding vectors...')
    index_ivf.add(combined_embeddings)
    IVF_BUILD_TIME = time.time() - t0
    faiss.write_index(index_ivf, str(ivf_path))
    print(f'[Index:IVF] Built in {IVF_BUILD_TIME:.1f}s, {index_ivf.ntotal:,} vectors, nlist={IVF_NLIST}')

[Index:IVF] Loading cached IVF index...


In [13]:
# ── HNSW (CPU only -- FAISS has no GPU HNSW implementation) ────────────────
HNSW_M = 32              # graph connectivity -- higher = better recall, more memory
HNSW_EF_CONSTRUCTION = 200  # build-time search depth -- higher = better graph, slower build

hnsw_path = RESULT_DIR / 'index_hnsw.faiss'
if hnsw_path.exists():
    print('[Index:HNSW] Loading cached HNSW index...')
    index_hnsw = faiss.read_index(str(hnsw_path))
    HNSW_BUILD_TIME = None
else:
    print(f'[Index:HNSW] Building IndexHNSWFlat (M={HNSW_M}, efConstruction={HNSW_EF_CONSTRUCTION})...')
    print('[Index:HNSW] This runs on CPU and is typically the slowest index to build -- '
          'reduce TARGET_CORPUS_SIZE in Section 3 if this takes too long for your job time limit.')
    t0 = time.time()
    index_hnsw = faiss.IndexHNSWFlat(DIM, HNSW_M)
    index_hnsw.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
    index_hnsw.add(combined_embeddings)
    HNSW_BUILD_TIME = time.time() - t0
    faiss.write_index(index_hnsw, str(hnsw_path))
    print(f'[Index:HNSW] Built in {HNSW_BUILD_TIME/60:.1f} minutes, {index_hnsw.ntotal:,} vectors')

[Index:HNSW] Building IndexHNSWFlat (M=32, efConstruction=200)...
[Index:HNSW] This runs on CPU and is typically the slowest index to build -- reduce TARGET_CORPUS_SIZE in Section 3 if this takes too long for your job time limit.
[Index:HNSW] Built in 43.5 minutes, 2,000,000 vectors


In [14]:
def index_file_size_mb(path):
    return Path(path).stat().st_size / 1e6

print('[Index] ============================================================')
print('[Index] INDEX FOOTPRINT (on-disk size, as a proxy for memory usage)')
print('[Index] ============================================================')
print(f'  Exact (FlatL2) : {index_file_size_mb(flat_path):>10.1f} MB')
print(f'  IVF            : {index_file_size_mb(ivf_path):>10.1f} MB')
print(f'  HNSW           : {index_file_size_mb(hnsw_path):>10.1f} MB  '
      f'(graph edges make this the largest -- expected)')

[Index] ============================================================
[Index] INDEX FOOTPRINT (on-disk size, as a proxy for memory usage)
[Index] ============================================================
  Exact (FlatL2) :     3072.0 MB
  IVF            :     3094.3 MB
  HNSW           :     3616.3 MB  (graph edges make this the largest -- expected)


## 5 · Load Queries and Ground Truth

In [15]:
print('[Load] Loading queries...')
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries : {len(data)}')

print('[Load] Loading production ground-truth labels...')
production_df = pd.read_excel('dataset/production_results.xlsx')

K_VALUES = [10, 50, 100, 500, 1000]

def get_relevant(query_id, top_k=1000):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(1 / np.log2(i + 2) for i, d in enumerate(retrieved[:k]) if d in relevant)

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

print('[Eval] Metric functions defined ✅ (same protocol as 03_baseline_minilm.ipynb)')

[Load] Loading queries...
[Load] Queries : 101
[Load] Loading production ground-truth labels...
[Eval] Metric functions defined ✅ (same protocol as 03_baseline_minilm.ipynb)


## 6 · Load MiniLM Model (for Query Encoding)

In [16]:
from sentence_transformers import SentenceTransformer

print('[Model] Loading MiniLM (all-MiniLM-L6-v2)...')
t0 = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f'[Model] Loaded in {time.time()-t0:.1f}s on {model.device}')

[Model] Loading MiniLM (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Model] Loaded in 2.9s on cpu


## 7 · Run Retrieval for Each Index Configuration

Each configuration (exact, IVF at several `nprobe` values, HNSW at several
`efSearch` values) is run over all 101 queries. A retrieved FAISS index position
`>= N_REAL` is synthetic filler -- it's kept in the retrieved list (so ranking
position is realistic) but mapped to a domain that can never match ground truth,
exactly simulating what happens when most of a real large corpus is irrelevant
to any given query.

In [17]:
IVF_NPROBE_VALUES   = [1, 8, 32, 128, IVF_NLIST]  # IVF_NLIST = exhaustive-over-clusters upper bound
HNSW_EFSEARCH_VALUES = [16, 64, 256]

domain_lookup = all_companies['domain'].tolist()  # index position -> domain, for REAL indices only

def domain_for_idx(idx):
    return domain_lookup[idx] if idx < N_REAL else None  # None = synthetic, never relevant

def run_retrieval(index, config_name, set_param_fn=None):
    print(f'[Retrieval] Running {config_name}...')
    all_results  = []
    query_times  = []

    for item in data:
        query_id, query = item['query_id'], item['query']
        if set_param_fn:
            set_param_fn(index)

        t0 = time.perf_counter()
        query_emb = model.encode([query], normalize_embeddings=False, convert_to_numpy=True).astype('float32')
        distances, idxs = index.search(query_emb, 1000)
        query_ms = (time.perf_counter() - t0) * 1000
        query_times.append(query_ms)

        for rank, (idx, dist) in enumerate(zip(idxs[0], distances[0])):
            if idx < 0:  # FAISS returns -1 for HNSW when fewer than k results are found
                continue
            all_results.append({
                'config':    config_name,
                'query_id':  query_id,
                'rank':      rank + 1,
                'domain':    domain_for_idx(int(idx)),
            })

    avg_ms = sum(query_times) / len(query_times)
    print(f'[Retrieval] {config_name} done — avg {avg_ms:.2f}ms/query')
    return pd.DataFrame(all_results), avg_ms


all_configs_results = {}
all_configs_latency = {}

df, ms = run_retrieval(index_flat, 'Exact (FlatL2)')
all_configs_results['Exact (FlatL2)'] = df
all_configs_latency['Exact (FlatL2)'] = ms

for nprobe in IVF_NPROBE_VALUES:
    name = f'IVF (nprobe={nprobe})'
    df, ms = run_retrieval(index_ivf, name, set_param_fn=lambda idx, p=nprobe: setattr(idx, 'nprobe', p))
    all_configs_results[name] = df
    all_configs_latency[name] = ms

for ef in HNSW_EFSEARCH_VALUES:
    name = f'HNSW (efSearch={ef})'
    df, ms = run_retrieval(index_hnsw, name, set_param_fn=lambda idx, e=ef: setattr(idx.hnsw, 'efSearch', e))
    all_configs_results[name] = df
    all_configs_latency[name] = ms

print('[Retrieval] All configurations done ✅')

[Retrieval] Running Exact (FlatL2)...
[Retrieval] Exact (FlatL2) done — avg 175.46ms/query
[Retrieval] Running IVF (nprobe=1)...
[Retrieval] IVF (nprobe=1) done — avg 21.56ms/query
[Retrieval] Running IVF (nprobe=8)...
[Retrieval] IVF (nprobe=8) done — avg 26.37ms/query
[Retrieval] Running IVF (nprobe=32)...
[Retrieval] IVF (nprobe=32) done — avg 48.77ms/query
[Retrieval] Running IVF (nprobe=128)...
[Retrieval] IVF (nprobe=128) done — avg 108.45ms/query
[Retrieval] Running IVF (nprobe=4096)...
[Retrieval] IVF (nprobe=4096) done — avg 127.36ms/query
[Retrieval] Running HNSW (efSearch=16)...
[Retrieval] HNSW (efSearch=16) done — avg 14.36ms/query
[Retrieval] Running HNSW (efSearch=64)...
[Retrieval] HNSW (efSearch=64) done — avg 16.81ms/query
[Retrieval] Running HNSW (efSearch=256)...
[Retrieval] HNSW (efSearch=256) done — avg 26.93ms/query
[Retrieval] All configurations done ✅


## 8 · Evaluation

In [18]:
print('[Eval] Computing metrics for all configurations...')
eval_rows = []

for config_name, retrieved_df in all_configs_results.items():
    for item in data:
        qid = item['query_id']
        relevant = get_relevant(qid)
        retrieved = (
            retrieved_df[retrieved_df['query_id'] == qid]
            .sort_values('rank')['domain'].tolist()
        )
        for k in K_VALUES:
            eval_rows.append({
                'config':    config_name,
                'query_id':  qid,
                'k':         k,
                'precision': precision_at_k(retrieved, relevant, k),
                'recall':    recall_at_k(retrieved, relevant, k),
                'f1':        f1_at_k(retrieved, relevant, k),
                'ndcg':      ndcg_at_k(retrieved, relevant, k),
            })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_ann.csv', index=False)
print(f'[Eval] Done — {len(eval_df):,} rows saved to {RESULT_DIR}/evaluation_ann.csv')

[Eval] Computing metrics for all configurations...
[Eval] Done — 4,545 rows saved to result/11_ann_scaling/evaluation_ann.csv


## 9 · Results: Latency, Memory, and Quality Side by Side

In [19]:
build_times = {
    'Exact (FlatL2)': FLAT_BUILD_TIME,
    'IVF':            IVF_BUILD_TIME,
    'HNSW':           HNSW_BUILD_TIME,
}
footprints_mb = {
    'Exact (FlatL2)': index_file_size_mb(flat_path),
    'IVF':            index_file_size_mb(ivf_path),
    'HNSW':           index_file_size_mb(hnsw_path),
}

print('=' * 60)
print('INDEX-LEVEL EFFICIENCY (build time + memory footprint)')
print('=' * 60)
print(f'  {"Index":<16} {"Build time":>14} | {"Footprint (MB)":>15}')
print('  ' + '-' * 55)
for name in ['Exact (FlatL2)', 'IVF', 'HNSW']:
    bt = build_times[name]
    bt_str = f'{bt:.1f}s' if bt is not None else '(loaded from cache)'
    print(f'  {name:<16} {bt_str:>14} | {footprints_mb[name]:>15.1f}')

print('\n' + '=' * 90)
print(f'ANN EVALUATION  (corpus size = {N_TOTAL:,}, real labeled companies = {N_REAL:,})')
print('=' * 90)
print(f'  {"Config":<22} {"k":>6} | {"NDCG":>7} | {"Prec":>7} | {"Recall":>7} | {"F1":>7} | {"ms/query":>9}')
print('  ' + '=' * 90)

summary_rows = []
for config_name in all_configs_results:
    avg_ms = all_configs_latency[config_name]
    for k in K_VALUES:
        sub = eval_df[(eval_df['config'] == config_name) & (eval_df['k'] == k)]
        ndcg, prec, rec, f1 = sub['ndcg'].mean(), sub['precision'].mean(), sub['recall'].mean(), sub['f1'].mean()
        print(f'  {config_name:<22} {k:>6} | {ndcg:>7.3f} | {prec:>7.3f} | {rec:>7.3f} | {f1:>7.3f} | {avg_ms:>9.2f}')
        summary_rows.append({'config': config_name, 'k': k, 'ndcg': round(ndcg,4),
                              'precision': round(prec,4), 'recall': round(rec,4),
                              'f1': round(f1,4), 'avg_ms_per_query': round(avg_ms,2)})
    print('  ' + '-' * 90)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULT_DIR / 'latency_memory_summary.csv', index=False)

index_efficiency_df = pd.DataFrame([
    {'index': name, 'build_time_s': build_times[name], 'footprint_mb': round(footprints_mb[name], 1)}
    for name in ['Exact (FlatL2)', 'IVF', 'HNSW']
])
index_efficiency_df.to_csv(RESULT_DIR / 'index_efficiency.csv', index=False)
print(f'\n[Results] Saved to {RESULT_DIR}/latency_memory_summary.csv and {RESULT_DIR}/index_efficiency.csv')

INDEX-LEVEL EFFICIENCY (build time + memory footprint)
  Index                Build time |  Footprint (MB)
  -------------------------------------------------------
  Exact (FlatL2)   (loaded from cache) |          3072.0
  IVF              (loaded from cache) |          3094.3
  HNSW                    2612.6s |          3616.3

ANN EVALUATION  (corpus size = 2,000,000, real labeled companies = 98,716)
  Config                      k |    NDCG |    Prec |  Recall |      F1 |  ms/query
  Exact (FlatL2)             10 |   0.975 |   0.972 |   0.010 |   0.019 |    175.46
  Exact (FlatL2)             50 |   0.955 |   0.950 |   0.047 |   0.090 |    175.46
  Exact (FlatL2)            100 |   0.945 |   0.938 |   0.094 |   0.171 |    175.46
  Exact (FlatL2)            500 |   0.872 |   0.858 |   0.429 |   0.572 |    175.46
  Exact (FlatL2)           1000 |   0.767 |   0.741 |   0.741 |   0.741 |    175.46
  ---------------------------------------------------------------------------------------

## 10 · Key Findings

In [20]:
print('[Findings] ============================================================')
print('[Findings] EXACT vs ANN AT ~{:,} VECTORS'.format(N_TOTAL))
print('[Findings] ============================================================')

exact_ms = all_configs_latency['Exact (FlatL2)']
print(f'\n  Exact (FlatL2) latency : {exact_ms:.2f}ms/query  (brute-force over {N_TOTAL:,} vectors)')

for n in all_configs_latency:
    if n == 'Exact (FlatL2)':
        continue
    ms = all_configs_latency[n]
    speedup = exact_ms / ms if ms > 0 else float('inf')
    ndcg10 = eval_df[(eval_df['config']==n) & (eval_df['k']==10)]['ndcg'].mean()
    rec1000 = eval_df[(eval_df['config']==n) & (eval_df['k']==1000)]['recall'].mean()
    exact_ndcg10 = eval_df[(eval_df['config']=='Exact (FlatL2)') & (eval_df['k']==10)]['ndcg'].mean()
    exact_rec1000 = eval_df[(eval_df['config']=='Exact (FlatL2)') & (eval_df['k']==1000)]['recall'].mean()
    print(f'\n  {n}:')
    print(f'    Latency : {ms:.2f}ms/query  ({speedup:.1f}x {"faster" if speedup > 1 else "slower"} than exact)')
    print(f'    NDCG@10 : {ndcg10:.3f}  (exact: {exact_ndcg10:.3f})')
    print(f'    Recall@1000 : {rec1000:.3f}  (exact: {exact_rec1000:.3f})')

print('\n[Findings] ============================================================')
print('[Findings] REMINDER OF SCOPE')
print('[Findings] ============================================================')
print(f'  Corpus: {N_REAL:,} real, labeled companies + {N_TOTAL-N_REAL:,} synthetic filler vectors.')
print('  Latency numbers reflect a realistic ~{:,}-vector scale. Quality numbers show'.format(N_TOTAL))
print('  whether ANN preserves exact search\'s ability to find the REAL companies once')
print('  the index is padded to that scale -- NOT a test against an independently-sourced,')
print('  truly 20M-scale real corpus (no such corpus was available locally for this thesis).')
print('\n[Done] result/11_ann_scaling/ — all files saved.')

[Findings] ============================================================
[Findings] EXACT vs ANN AT ~2,000,000 VECTORS
[Findings] ============================================================

  Exact (FlatL2) latency : 175.46ms/query  (brute-force over 2,000,000 vectors)

  IVF (nprobe=1):
    Latency : 21.56ms/query  (8.1x faster than exact)
    NDCG@10 : 0.981  (exact: 0.975)
    Recall@1000 : 0.641  (exact: 0.741)

  IVF (nprobe=8):
    Latency : 26.37ms/query  (6.7x faster than exact)
    NDCG@10 : 0.974  (exact: 0.975)
    Recall@1000 : 0.743  (exact: 0.741)

  IVF (nprobe=32):
    Latency : 48.77ms/query  (3.6x faster than exact)
    NDCG@10 : 0.975  (exact: 0.975)
    Recall@1000 : 0.740  (exact: 0.741)

  IVF (nprobe=128):
    Latency : 108.45ms/query  (1.6x faster than exact)
    NDCG@10 : 0.975  (exact: 0.975)
    Recall@1000 : 0.741  (exact: 0.741)

  IVF (nprobe=4096):
    Latency : 127.36ms/query  (1.4x faster than exact)
    NDCG@10 : 0.975  (exact: 0.975)
    Recall@1000 